# Data Investigation

## 1.1 Load Data from Huggingface

### Install modules

In [38]:
# generic requirements
!pip install ipywidgets jupyterlab

In [ ]:
# project requirements
!pip install datasets scikit-learn matplotlib seaborn eli5 google-cloud google-cloud-aiplatform kfp

  Using cached kfp-2.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached click_option_group-0.5.7-py3-none-any.whl.metadata (5.8 kB)
  Using cached kfp_pipeline_spec-2.16.0-py3-none-any.whl.metadata (433 bytes)
  Using cached kfp_server_api-2.16.0.tar.gz (63 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached kubernetes-30.1.0-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
Using cached kfp-2.16.0-py3-none-any.whl (418 kB)
Using cached click_option_group-0.5.7-py3-none-any.whl (11 kB)
Using cached kfp_pipeline_spec-2.16.0-py3-none-any.whl (9.9 kB)
Using cached kubernetes-30.1.0-py2.py3-none-any.whl (1.7 MB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Usin

### Load Data

In [40]:
from datasets import load_dataset

# Stream data 
ds = load_dataset("Tobi-Bueck/customer-support-tickets", streaming=True)

# Look at a few rows
for row in ds["train"].take(3):
    print(row)

{'subject': 'Wesentlicher Sicherheitsvorfall', 'body': 'Sehr geehrtes Support-Team,\\n\\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\\n\\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnte die Bedrohung bislang nicht vollständig eliminiert.', 'answer': 'Vielen Dank für die Meldung des kritischen Sicherheitsvorfalls und die Bereitstellung der Übersicht über die betroffenen Geräte sowie der ergriffenen ersten Maßnahmen. Wir erk

In [41]:
# Load entire dataset without splits
ds = load_dataset("Tobi-Bueck/customer-support-tickets")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8'],
        num_rows: 61765
    })
})


In [42]:
# Convert to Pandas for analysis
df = ds["train"].to_pandas()
df = df[df["language"] == "en"]
print(df.shape)
print(df.dtypes)
df.head()

(28261, 16)
subject         str
body            str
answer          str
type            str
queue           str
priority        str
language        str
version     float64
tag_1           str
tag_2           str
tag_3           str
tag_4           str
tag_5           str
tag_6           str
tag_7           str
tag_8           str
dtype: object


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN


# 1.2 Basic Analysis of Dataset

Initial review of data. For this project it will focus on English language only.

### Distribution of ticket types


In [43]:
df["type"].value_counts()


type
Incident    11213
Request      8163
Problem      5895
Change       2990
Name: count, dtype: int64

### Queue Types

In [44]:
df["queue"].value_counts()

queue
Technical Support                  8149
Product Support                    5305
Customer Service                   4269
IT Support                         3333
Billing and Payments               2897
Returns and Exchanges              1402
Service Outages and Maintenance    1106
Sales and Pre-Sales                 843
Human Resources                     553
General Inquiry                     404
Name: count, dtype: int64

### Priority breakdown

In [45]:
df["priority"].value_counts()

priority
medium    11570
high      10917
low        5774
Name: count, dtype: int64

### Missing values across tag columns

In [46]:
df[["tag_1","tag_2","tag_3","tag_4","tag_5"]].isna().sum()

tag_1        0
tag_2       16
tag_3      113
tag_4     2584
tag_5    11960
dtype: int64

## 2.1 Data Readiness

### Drop NA from Subject or Body

In [47]:
df = df.dropna(subset=["subject", "body"])

### Combine Subject and Body


In [48]:
df["combined"] = df["subject"] + " " + df["body"]
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,combined
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,"Account Disruption Dear Customer Support Team,..."
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,Query About Smart Home System Integration Feat...
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,Inquiry Regarding Invoice Details Dear Custome...
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,Question About Marketing Agency Software Compa...
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN,"Feature Query Dear Customer Support,\n\nI hope..."


### Clean Combination

Removes salutations. Lowecase. Remove newline. Collapse whitespace.

In [49]:
import re

def clean_text(text):
    text = text.replace("\\n", " ")
    text = re.sub(r"Dear[^,]+,", "", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Before
print("BEFORE:", df["combined"].iloc[0][:200])
print("---")

# Apply cleaning
df["combined"] = df["combined"].apply(clean_text)

# After
print("AFTER:", df["combined"].iloc[0][:200])

BEFORE: Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blo
---
AFTER: account disruption i am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. this outage is blocking access to account setting


## 2.2 Data Split

Stratified sampling and split 80:10:10 (train, validate, test)

In [50]:
from sklearn.model_selection import train_test_split

X = df["combined"]
y = df["queue"]

# First split: 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Second split: split temp 50/50 into val and test (10% each of original)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

Verify split distribution across the three datasets

In [51]:
import pandas as pd

pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "val": y_val.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True)
})

,train,val,test
queue,,,
Technical Support,0.287977,0.287977,0.287860
Product Support,0.188566,0.188465,0.188794
Customer Service,0.152518,0.152315,0.152659
IT Support,0.117080,0.116978,0.117337
Billing and Payments,0.103270,0.103168,0.103532
Returns and Exchanges,0.049401,0.049553,0.049127
Service Outages and Maintenance,0.040465,0.040617,0.040195
Sales and Pre-Sales,0.028128,0.028026,0.028015
Human Resources,0.018989,0.019090,0.019082


## 3.0 Local Training

TF-IDF

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Fit TF-IDF on training data only
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

Logistic Regression

In [53]:
%%time

logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(X_train_tfidf, y_train)

CPU times: user 3.66 s, sys: 31.2 ms, total: 3.69 s
Wall time: 3.7 s


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

Predict on Validation Set

In [54]:
from sklearn.metrics import classification_report

y_val_pred = logit.predict(X_val_tfidf)
print(classification_report(y_val, y_val_pred))

                                 precision    recall  f1-score   support

           Billing and Payments       0.88      0.73      0.80       254
               Customer Service       0.42      0.44      0.43       375
                General Inquiry       0.00      0.00      0.00        34
                Human Resources       1.00      0.15      0.26        47
                     IT Support       0.58      0.32      0.41       288
                Product Support       0.49      0.50      0.49       464
          Returns and Exchanges       0.80      0.13      0.23       122
            Sales and Pre-Sales       0.82      0.13      0.23        69
Service Outages and Maintenance       0.85      0.47      0.61       100
              Technical Support       0.51      0.81      0.62       709

                       accuracy                           0.54      2462
                      macro avg       0.63      0.37      0.41      2462
                   weighted avg       0.58      0

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/james.carty/Documents/VScode/google-ml-engineer/.venv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/james.carty/Documents/VScode/google-ml-engineer/.venv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_d

Redo with Class Weights to penalise mistakes on small class more heavily

In [55]:
logit_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
logit_balanced.fit(X_train_tfidf, y_train)

y_val_pred_balanced = logit_balanced.predict(X_val_tfidf)
print(classification_report(y_val, y_val_pred_balanced))

                                 precision    recall  f1-score   support

           Billing and Payments       0.77      0.76      0.76       254
               Customer Service       0.42      0.39      0.40       375
                General Inquiry       0.26      0.50      0.34        34
                Human Resources       0.46      0.70      0.55        47
                     IT Support       0.44      0.51      0.47       288
                Product Support       0.55      0.41      0.47       464
          Returns and Exchanges       0.39      0.59      0.47       122
            Sales and Pre-Sales       0.30      0.77      0.43        69
Service Outages and Maintenance       0.50      0.72      0.59       100
              Technical Support       0.63      0.47      0.54       709

                       accuracy                           0.51      2462
                      macro avg       0.47      0.58      0.50      2462
                   weighted avg       0.54      0

Review a few classes to see how differentiated the data is

In [56]:
# Compare tickets from two classes that might be confusable
for queue in ["Customer Service", "General Inquiry"]:
    print(f"\n=== {queue} ===")
    samples = df[df["queue"] == queue]["combined"].head(3)
    for s in samples:
        print(s[:150])
        print("---")


=== Customer Service ===
inquiry for in-depth details on financial institution offerings i hope this message reaches you in good health. i am writing to request detailed infor
---
inquiry for comprehensive marketing service details i hope this message reaches you well. i am writing to request detailed information about your mark
---
intermittent access problems on saas platform currently facing sporadic connectivity difficulties with the cloud-native saas system. the suspected rea
---

=== General Inquiry ===
organizational revision i am reaching out to request an update on the structural details of our organization. the current records reflect information 
---
inquiry about company offerings i am seeking comprehensive details regarding the company's products and services. would you be able to send brochures,
---
question about service functionality i am reaching out to seek clarification on certain aspects of the current service operations that i find unclear.
---


## 4.0 Setup GCS and upload data

Setup google project

In [57]:
import os
from datetime import datetime

# --- Project Configuration ---
PROJECT_ID = "carty-470812"
REGION = "us-central1"
BUCKET_NAME = f"{PROJECT_ID}"
BUCKET_URI = f"gs://{BUCKET_NAME}"
PIPELINE_ROOT = f"{BUCKET_URI}/project1-support-tickets"

# Timestamp for unique resource naming (hyphens, not underscores — Vertex AI requirement)
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"Project:        {PROJECT_ID}")
print(f"Region:         {REGION}")
print(f"Pipeline Root:  {PIPELINE_ROOT}")
print(f"Serving Image:  {SKLEARN_SERVING_IMAGE}")
print(f"Component Deps: {COMPONENT_DEPS}")
print(f"Timestamp:      {TIMESTAMP}")

Project:        carty-470812
Region:         us-central1
Pipeline Root:  gs://carty-470812/project1-support-tickets
Serving Image:  us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest
Component Deps: ['scikit-learn==1.3.2', 'pandas', 'joblib', 'google-cloud-storage']
Timestamp:      20260329-082825


Initialise tooling

In [58]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

print("Clients initialized.")

Clients initialized.


Create dataset files and upload

In [59]:
train_df = pd.DataFrame({"text": X_train, "queue": y_train})
val_df = pd.DataFrame({"text": X_val, "queue": y_val})
test_df = pd.DataFrame({"text": X_test, "queue": y_test})

for name, data in [("train", train_df), ("val", val_df), ("test", test_df)]:
    filename = f"{name}.csv"
    data.to_csv(filename, index=False)
    !gsutil cp {filename} {PIPELINE_ROOT}/data/

Copying file://train.csv [Content-Type=text/csv]...
| [1 files][  8.0 MiB/  8.0 MiB]                                                
Operation completed over 1 objects/8.0 MiB.                                      
Copying file://val.csv [Content-Type=text/csv]...
- [1 files][ 1023 KiB/ 1023 KiB]                                                
Operation completed over 1 objects/1023.3 KiB.                                   
Copying file://test.csv [Content-Type=text/csv]...
- [1 files][ 1019 KiB/ 1019 KiB]                                                
Operation completed over 1 objects/1019.5 KiB.                                   


## 5.0 Pipeline

In [61]:
from kfp import dsl, compiler
from kfp.dsl import component

Training script

In [62]:
COMPONENT_DEPS = ["scikit-learn==1.3.2", "pandas", "joblib", "google-cloud-storage"]

@component(
    base_image="python:3.11",
    packages_to_install=COMPONENT_DEPS,
)
def train_model(
    train_gcs_path: str,
    model_gcs_path: str,
    max_iter: int = 1000,
    random_state: int = 42,
) -> str:
    """Train TF-IDF + LogisticRegression on training data."""
    import pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.feature_extraction.text import TfidfVectorizer
    import joblib
    from google.cloud import storage

    # 1. Download training CSV from GCS
    client = storage.Client()
    bucket_name = train_gcs_path.replace("gs://", "").split("/")[0]
    blob_path = "/".join(train_gcs_path.replace("gs://", "").split("/")[1:])
    bucket = client.bucket(bucket_name)
    bucket.blob(blob_path).download_to_filename("/tmp/train.csv")

    # 2. Load data
    train_df = pd.read_csv("/tmp/train.csv")
    X_train = train_df["text"]
    y_train = train_df["queue"]
    print(f"Training on {len(X_train)} samples")

    # 3. Fit TF-IDF
    tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    X_train_tfidf = tfidf.fit_transform(X_train)
    print(f"TF-IDF features: {X_train_tfidf.shape[1]}")

    # 4. Train LogReg
    model = LogisticRegression(
        max_iter=max_iter,
        random_state=random_state,
        class_weight="balanced",
    )
    model.fit(X_train_tfidf, y_train)
    train_accuracy = model.score(X_train_tfidf, y_train)
    print(f"Train accuracy: {train_accuracy:.4f}")

    # 5. Save model and vectorizer locally
    joblib.dump(model, "/tmp/model.joblib")
    joblib.dump(tfidf, "/tmp/tfidf.joblib")

    # 6. Upload both to GCS
    model_blob = model_gcs_path.replace("gs://", "").split("/", 1)[1]
    tfidf_blob = model_blob.rsplit("/", 1)[0] + "/tfidf.joblib"

    bucket.blob(model_blob).upload_from_filename("/tmp/model.joblib")
    bucket.blob(tfidf_blob).upload_from_filename("/tmp/tfidf.joblib")
    print(f"Model saved to {model_gcs_path}")
    print(f"Vectorizer saved to gs://{bucket_name}/{tfidf_blob}")

    return model_gcs_path

Evaluate

In [63]:
@component(
    base_image="python:3.11",
    packages_to_install=COMPONENT_DEPS,
)
def evaluate_model(
    val_gcs_path: str,
    model_gcs_path: str,
) -> float:
    """Evaluate the model on the validation set. Returns weighted F1."""
    import pandas as pd
    from sklearn.metrics import f1_score, classification_report
    import joblib
    from google.cloud import storage

    # 1. Download val CSV from GCS
    client = storage.Client()
    bucket_name = val_gcs_path.replace("gs://", "").split("/")[0]
    bucket = client.bucket(bucket_name)

    val_blob = "/".join(val_gcs_path.replace("gs://", "").split("/")[1:])
    bucket.blob(val_blob).download_to_filename("/tmp/val.csv")

    # 2. Download model and vectorizer from GCS
    model_blob = model_gcs_path.replace("gs://", "").split("/", 1)[1]
    tfidf_blob = model_blob.rsplit("/", 1)[0] + "/tfidf.joblib"

    bucket.blob(model_blob).download_to_filename("/tmp/model.joblib")
    bucket.blob(tfidf_blob).download_to_filename("/tmp/tfidf.joblib")

    # 3. Load everything
    val_df = pd.read_csv("/tmp/val.csv")
    X_val = val_df["text"]
    y_val = val_df["queue"]
    model = joblib.load("/tmp/model.joblib")
    tfidf = joblib.load("/tmp/tfidf.joblib")
    print(f"Evaluating on {len(X_val)} samples")

    # 4. Transform and predict
    X_val_tfidf = tfidf.transform(X_val)
    y_pred = model.predict(X_val_tfidf)

    # 5. Compute metrics
    weighted_f1 = f1_score(y_val, y_pred, average="weighted")
    print(classification_report(y_val, y_pred))
    print(f"Weighted F1: {weighted_f1:.4f}")

    return weighted_f1

Pipeline

In [65]:
@component(base_image="python:3.11")
def deploy_model(model_gcs_path: str):
    """Mock deployment — logs intent to deploy."""
    print(f"Would deploy model from {model_gcs_path}")

In [66]:
@dsl.pipeline(
    name="support-ticket-routing",
    pipeline_root=PIPELINE_ROOT,
)
def ticket_routing_pipeline(
    train_gcs_path: str,
    val_gcs_path: str,
    model_gcs_path: str,
    f1_threshold: float = 0.45,
):
    # Step 1: Train
    train_task = train_model(
        train_gcs_path=train_gcs_path,
        model_gcs_path=model_gcs_path,
    )

    # Step 2: Evaluate (waits for training to finish)
    eval_task = evaluate_model(
        val_gcs_path=val_gcs_path,
        model_gcs_path=model_gcs_path,
    ).after(train_task)

    # Step 3: Conditional deploy
    with dsl.Condition(eval_task.output >= f1_threshold, name="f1-check"):
        deploy_model(
            model_gcs_path=model_gcs_path,
        )

/var/folders/gm/vjgn8n7j4jq6v03c3nq66t8m0000gn/T/ipykernel_81480/635358519.py:24: DeprecationWarning: dsl.Condition is deprecated. Please use dsl.If instead.
  with dsl.Condition(eval_task.output >= f1_threshold, name="f1-check"):


Compile and Submit

In [67]:
from kfp import compiler

# Compile
compiler.Compiler().compile(
    pipeline_func=ticket_routing_pipeline,
    package_path="ticket_routing_pipeline.json",
)

# Submit
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION)

job = aiplatform.PipelineJob(
    display_name="support-ticket-routing",
    template_path="ticket_routing_pipeline.json",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "train_gcs_path": f"{PIPELINE_ROOT}/data/train.csv",
        "val_gcs_path": f"{PIPELINE_ROOT}/data/val.csv",
        "model_gcs_path": f"{PIPELINE_ROOT}/model/model.joblib",
    },
)

job.run(service_account=None)

Creating PipelineJob
PipelineJob created. Resource name: projects/873708835509/locations/us-central1/pipelineJobs/support-ticket-routing-20260329083059
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/873708835509/locations/us-central1/pipelineJobs/support-ticket-routing-20260329083059')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/support-ticket-routing-20260329083059?project=873708835509
PipelineJob projects/873708835509/locations/us-central1/pipelineJobs/support-ticket-routing-20260329083059 current state:
2
PipelineJob projects/873708835509/locations/us-central1/pipelineJobs/support-ticket-routing-20260329083059 current state:
2
PipelineJob projects/873708835509/locations/us-central1/pipelineJobs/support-ticket-routing-20260329083059 current state:
2
PipelineJob projects/873708835509/locations/us-central1/pipelineJobs/support-ticket-routing-20260329083059 current state:
3
PipelineJo

### 6.0 Local Confidence Routing


In [69]:
# Confidence-based routing on test set
from sklearn.metrics import classification_report
import numpy as np

# Use the locally trained balanced model (already in memory)
X_test_tfidf = tfidf.transform(X_test)
y_pred = logit_balanced.predict(X_test_tfidf)
y_proba = logit_balanced.predict_proba(X_test_tfidf)
confidence = np.max(y_proba, axis=1)

# Flag low-confidence predictions for human review
CONFIDENCE_THRESHOLD = 0.5
needs_review = confidence < CONFIDENCE_THRESHOLD

print(f"Total predictions: {len(y_pred)}")
print(f"Auto-routed (confidence >= {CONFIDENCE_THRESHOLD}): {(~needs_review).sum()} ({(~needs_review).mean():.1%})")
print(f"Flagged for human review: {needs_review.sum()} ({needs_review.mean():.1%})")
print(f"\n--- Auto-routed accuracy ---")
print(classification_report(y_test[~needs_review], y_pred[~needs_review]))
print(f"--- Flagged for review accuracy ---")
print(classification_report(y_test[needs_review], y_pred[needs_review]))

Total predictions: 2463
Auto-routed (confidence >= 0.5): 378 (15.3%)
Flagged for human review: 2085 (84.7%)

--- Auto-routed accuracy ---
                                 precision    recall  f1-score   support

           Billing and Payments       0.99      0.99      0.99       163
               Customer Service       1.00      0.70      0.82        10
                General Inquiry       0.92      0.92      0.92        12
                Human Resources       0.86      1.00      0.92        12
                     IT Support       0.74      0.74      0.74        19
                Product Support       0.94      0.76      0.84        21
          Returns and Exchanges       0.86      0.95      0.90        20
            Sales and Pre-Sales       0.81      0.96      0.88        23
Service Outages and Maintenance       0.76      1.00      0.86        60
              Technical Support       0.83      0.39      0.54        38

                       accuracy                          

## Project Summary

### Results

| Stage | Metric | Value |
|-------|--------|-------|
| Local Baseline (no class weights) | Accuracy | 54% |
| Local Baseline (no class weights) | Weighted F1 | 0.52 |
| Local Baseline (balanced weights) | Accuracy | 51% |
| Local Baseline (balanced weights) | Weighted F1 | 0.51 |
| Pipeline (balanced weights) | Weighted F1 | 0.5156 |
| Confidence Routing (auto-routed, ≥0.5) | Accuracy | 89% |
| Confidence Routing (auto-routed, ≥0.5) | Weighted F1 | 0.88 |
| Confidence Routing (flagged for review) | Accuracy | 44% |

### Key Learnings

- **Synthetic data sets a ceiling:** ambiguous text between classes (e.g. Customer Service vs General Inquiry) limits model performance regardless of algorithm or tuning. No model can separate what humans can't distinguish.
- **Class weighting trades overall accuracy for balanced coverage:** macro F1 improved from 0.41 → 0.50 with `class_weight='balanced'`, surfacing predictions for all 10 classes instead of ignoring the small ones.
- **Per-class classification reports are essential:** overall accuracy (54%) hid the fact that 4 classes had near-zero recall without class weighting.
- **Pipeline reproduced local results:** weighted F1 matched between local (0.51) and pipeline (0.5156), confirming the pipeline is correctly replicating the workflow.
- **Confidence routing works even on weak models:** auto-routing the top 15% most confident predictions lifted accuracy from 51% → 89%. The model knows what it doesn't know.
- **Agent-assigned fields are leakage:** tags, type, and answer are assigned after reading the ticket — using them as features would leak information unavailable at prediction time.

### Exam Patterns Reinforced

- **`dsl.Condition`** for conditional deployment based on metric thresholds
- **`.after(task)`** to enforce ordering when components share GCS paths rather than KFP artifacts
- **`predict_proba()` + confidence threshold** for human-in-the-loop routing
- **`class_weight='balanced'`** to handle imbalanced classes without oversampling
- **Stratified splitting** to preserve class distribution across train/val/test
- **TF-IDF fit on train only**, transform on val/test to prevent data leakage